In [22]:
import pandas as pd
import requests
import time
from pathlib import Path
import os
from io import StringIO
from rapidfuzz import process, fuzz

In [3]:
def parse_fwf_from_dashes(filepath: Path) -> pd.DataFrame:
    with open(filepath, "r") as f:
        lines = f.readlines()

    # second row (index 1) contains the dashed separators
    dash_line = lines[1]

    # derive column specs from contiguous dash groups
    colspecs = []
    start = None
    for i, ch in enumerate(dash_line):
        if ch == "-" and start is None:
            start = i
        elif ch != "-" and start is not None:
            colspecs.append((start, i))
            start = None
    if start is not None:
        colspecs.append((start, len(dash_line)))

    # replace header with actual names from row 0
    header_line = lines[0]
    col_names = [header_line[s:e].strip() for s, e in colspecs]

    emshr_df = pd.read_fwf(
        filepath,
        colspecs=colspecs,
        names=col_names,
        skiprows=2       # skip header row and dash row
    )

    print(f"EMSHR dataframe shape: {emshr_df.shape}")
    return emshr_df

In [4]:

emshr_df = parse_fwf_from_dashes("emshr_lite_202602.txt")

EMSHR dataframe shape: (308213, 33)


In [14]:
emshr_df = emshr_df[emshr_df["WBAN"].notna()]
emshr_df = emshr_df.drop_duplicates(subset="GHCND")
emshr_df = emshr_df[emshr_df["GHCND"].notna()]
emshr_df = emshr_df[["GHCND", "STATION_NAME", "ST", "COUNTY", "LAT_DEC", "LON_DEC"]]

In [15]:
emshr_df.to_csv("all_wban_us_weather_stations.csv", index=False)

In [5]:
weather_stations = pd.read_csv("all_wban_us_weather_stations.csv")

In [6]:
# Base URL template with placeholder for station ID
base_url = (
    "https://www.ncei.noaa.gov/access/services/data/v1"
    "?dataset=global-summary-of-the-year"
    "&stations={station_id}"
    "&startDate=2012-01-01"
    "&endDate=2023-12-31"
    "&dataTypes=CLDD,DT100,DT32,DX32,DX70,DX90,EMNT,EMXT,HTDD,PRCP,TAVG,TMAX,TMIN"
    "&format=csv"
    "&includeStationName=true"
    "&includeStationLocation=1"
    "&units=metric"
)

In [7]:
all_data = []
output_file = "combined_weather_raw.csv"
first_write = not os.path.exists(output_file)

for station_id in weather_stations["GHCND"]:
    url = base_url.format(station_id=station_id)
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        df = pd.read_csv(StringIO(response.text))
        
        # Append to CSV incrementally
        df.to_csv(output_file, mode="a", header=first_write, index=False)
        first_write = False
        print(f"Fetched data for station: {station_id}")
        
    except requests.exceptions.RequestException as e:
        print(f"Failed for station {station_id}: {e}")
    
    time.sleep(0.5)

Fetched data for station: USW00024285
Fetched data for station: USW00025322
Fetched data for station: USW00025325
Fetched data for station: USC00506093
Fetched data for station: USW00025335
Fetched data for station: USC00509919
Fetched data for station: USW00053864
Fetched data for station: USW00013896
Fetched data for station: USW00093806
Fetched data for station: USC00030936
Fetched data for station: USW00023104
Fetched data for station: USW00003195
Fetched data for station: USC00024639
Fetched data for station: USW00093167
Fetched data for station: USW00093027
Fetched data for station: USW00023191
Fetched data for station: USW00023161
Fetched data for station: USW00093193
Fetched data for station: USW00023179
Fetched data for station: USW00024216
Fetched data for station: USC00047296
Fetched data for station: USW00024257
Fetched data for station: USC00043410
Fetched data for station: USW00003131
Fetched data for station: USW00023188
Fetched data for station: USW00093107
Fetched data

In [8]:
combined_df = pd.read_csv("combined_weather_raw.csv")
combined_df_new = combined_df.merge(
    weather_stations[["GHCND", "COUNTY", "ST"]],
    left_on="STATION",
    right_on="GHCND",
    how="left"
).drop(columns="GHCND")  # drop duplicate ID column after merge

combined_df_new.head()

,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,DATE,CLDD,DT100,DT32,DX32,...,DX90,EMNT,EMXT,HTDD,PRCP,TAVG,TMAX,TMIN,COUNTY,ST
0,USW00025322,"GUSTAVUS, AK US",58.4111,-135.7089,12.2,2012,0.0,NaN,191.0,54.0,...,0.0,-19.4,30.0,5005.8,1496.6,4.0,7.7,0.2,SKAGWAY-HOONAH-ANGOON,AK
1,USW00025322,"GUSTAVUS, AK US",58.4111,-135.7089,12.2,2013,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,5209.6,NaN,NaN,NaN,NaN,SKAGWAY-HOONAH-ANGOON,AK
2,USW00025322,"GUSTAVUS, AK US",58.4111,-135.7089,12.2,2014,0.0,NaN,175.0,41.0,...,0.0,-21.1,24.4,NaN,1516.3,4.8,9.1,0.4,SKAGWAY-HOONAH-ANGOON,AK
3,USW00025322,"GUSTAVUS, AK US",58.4111,-135.7089,12.2,2015,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,4656.6,NaN,NaN,NaN,NaN,SKAGWAY-HOONAH-ANGOON,AK
4,USW00025322,"GUSTAVUS, AK US",58.4111,-135.7089,12.2,2016,0.0,NaN,151.0,30.0,...,0.0,-18.9,28.9,NaN,1457.3,6.0,10.2,1.8,SKAGWAY-HOONAH-ANGOON,AK


In [13]:
missing_county_new = combined_df_new[combined_df_new["COUNTY"].isna()]

print(f"Rows with missing county: {len(missing_county_new)}")
print(f"Unique stations with missing county: {missing_county_new['STATION'].nunique()}")
print("\nStations with missing county:")
print(missing_county_new[["STATION", "ST", "COUNTY", "NAME"]].drop_duplicates())

missing_stations = combined_df_new[combined_df_new["COUNTY"].isna()][["STATION", "NAME", "ST"]].drop_duplicates()
missing_stations.to_csv("missing_county_stations.csv", index=False)
print(f"Exported {len(missing_stations)} stations missing county names")

Rows with missing county: 221
Unique stations with missing county: 29

Stations with missing county:
           STATION   ST COUNTY                            NAME
1448   USC00505778   AK    NaN            MCKINLEY PARK, AK US
9460   USC00502587   AK    NaN             DUTCH HARBOR, AK US
9812   USC00505881   AK    NaN               MINCHUMINA, AK US
9824   USW00026415   AK    NaN        BIG DELTA AIRPORT, AK US
9932   USC00505644   AK    NaN       MANLEY HOT SPRINGS, AK US
10024  USW00027502   AK    NaN           BARROW AIRPORT, AK US
11963  CJW00011813  NaN    NaN                     ROBERTS, CJ
11970  CUW00011706  NaN    NaN          GUANTANAMO BAY NAS, CU
11981  GMW00034068  NaN    NaN                     COLEMAN, GM
11993  GMW00034178  NaN    NaN            FURSTENFELDBRUCK, GM
12005  GMW00035032  NaN    NaN                  RHEIN MAIN, GM
12017  GLW00016504  NaN    NaN                 SONDRESTROM, GL
12025  GLW00017605  NaN    NaN                    PITUFFIK, GL
12029  ITW0003312

In [14]:
missing_county_names = pd.read_csv("missing_county_stations_updated.csv")

In [16]:
# Update missing county names in combined_df_new using the manually updated CSV
combined_df_new = combined_df_new.merge(
    missing_county_names[["STATION", "COUNTY"]],
    on="STATION",
    how="left",
    suffixes=("", "_updated")
)

# Fill in missing COUNTY values with the updated ones
combined_df_new["COUNTY"] = combined_df_new["COUNTY"].fillna(combined_df_new["COUNTY_updated"])

# Drop the temporary update columns
combined_df_new = combined_df_new.drop(columns=["COUNTY_updated"])

# Remove rows with missing county data
combined_df_new = combined_df_new.dropna(subset=["COUNTY"])

print(f"Remaining rows: {len(combined_df_new)}")
print(f"Remaining rows with missing county: {combined_df_new['COUNTY'].isna().sum()}")


Remaining rows: 16314
Remaining rows with missing county: 0


In [17]:
# Count unique weather stations per county
stations_per_county = combined_df_new.groupby(["ST", "COUNTY"])["STATION"].nunique().reset_index()
stations_per_county.columns = ["ST", "COUNTY", "num_stations"]

# Count entries per weather station
entries_per_station = combined_df_new.groupby(["ST", "COUNTY", "STATION"]).size().reset_index()
entries_per_station.columns = ["ST", "COUNTY", "STATION", "num_entries"]

print("Weather stations per county:")
print(stations_per_county.sort_values("num_stations", ascending=False))

print("\nEntries per weather station:")
print(entries_per_station.sort_values("num_entries", ascending=False))

Weather stations per county:
      ST               COUNTY  num_stations
120   CA          LOS ANGELES            13
133   CA            SAN DIEGO            12
30    AK        YUKON-KOYUKUK            12
19    AK  NORTH SLOPE BOROUGH             9
903   TX              TARRANT             8
...   ..                  ...           ...
520   MT      LEWIS AND CLARK             1
521   MT              MINERAL             1
522   MT             MISSOULA             1
523   MT                 PARK             1
1054  WY               WESTON             1

[1055 rows x 3 columns]

Entries per weather station:
      ST                  COUNTY      STATION  num_entries
0     AK  ALEUTIANS EAST BOROUGH  USW00025624           12
959   NM                 LINCOLN  USC00297649           12
956   NM                   GRANT  USW00093063           12
955   NM                    EDDY  USW00093033           12
953   NM                    EDDY  USW00003035           12
...   ..                     ...  

In [18]:
# Define the data columns to check for missing values
data_cols = ["CLDD", "DT100", "DT32", "DX32", "DX70", "DX90", "EMNT", "EMXT", "HTDD", "PRCP", "TAVG", "TMAX", "TMIN"]

# Filter to counties with more than 1 weather station
multi_station_counties = stations_per_county[stations_per_county["num_stations"] > 1][["ST", "COUNTY"]]

# Get only rows from counties with more than 1 station
df_multi = combined_df_new.merge(multi_station_counties, on=["ST", "COUNTY"], how="inner")

# Count rows and missing values per station across all data columns
station_quality = df_multi.groupby(["ST", "COUNTY", "STATION"]).apply(
    lambda x: pd.Series({
        "num_rows": len(x),
        "num_missing": x[data_cols].isnull().sum().sum(),
        "missing_rate": x[data_cols].isnull().sum().sum() / (len(x) * len(data_cols))
    })
).reset_index()

# Rank within each county: most rows first, fewest missing first
station_quality["row_rank"] = station_quality.groupby(["ST", "COUNTY"])["num_rows"].rank(ascending=False)
station_quality["missing_rank"] = station_quality.groupby(["ST", "COUNTY"])["missing_rate"].rank(ascending=True)

# Combined rank (lower is better)
station_quality["combined_rank"] = station_quality["row_rank"] + station_quality["missing_rank"]

# Best station per county
best_stations = station_quality.sort_values("combined_rank").groupby(["ST", "COUNTY"]).first().reset_index()

print("Best station per county (most rows, fewest missing):")
print(best_stations[["ST", "COUNTY", "STATION", "num_rows", "num_missing", "missing_rate"]]
      .sort_values(["ST", "COUNTY"]))

Best station per county (most rows, fewest missing):
     ST                  COUNTY      STATION  num_rows  num_missing  \
0    AK  ALEUTIANS EAST BOROUGH  USW00025624      12.0         12.0   
1    AK          ALEUTIANS WEST  USW00025713      12.0         45.0   
2    AK       ANCHORAGE BOROUGH  USW00026409      12.0         12.0   
3    AK                  BETHEL  USW00026615      12.0         13.0   
4    AK     BRISTOL BAY BOROUGH  USW00025506      12.0         24.0   
..   ..                     ...          ...       ...          ...   
280  WY                   CROOK  USW00094088      12.0         12.0   
281  WY                 FREMONT  USW00024021      12.0         32.0   
282  WY                 JOHNSON  USC00485055      12.0         12.0   
283  WY                    PARK  USC00489905      12.0         46.0   
284  WY                   TETON  USW00004131      12.0         12.0   

     missing_rate  
0        0.076923  
1        0.288462  
2        0.076923  
3        0.083

In [19]:
# Get single-station counties (already have only one station)
single_station_counties = stations_per_county[stations_per_county["num_stations"] == 1][["ST", "COUNTY"]]

# Get the best station IDs from multi-station counties
best_station_ids = best_stations[["ST", "COUNTY", "STATION"]]

# Get all stations from single-station counties
single_station_ids = combined_df_new.merge(single_station_counties, on=["ST", "COUNTY"], how="inner")[["ST", "COUNTY", "STATION"]].drop_duplicates()

# Combine both sets of stations to keep
stations_to_keep = pd.concat([best_station_ids, single_station_ids], ignore_index=True)

# Filter combined_df_new to only keep those stations
combined_df_final = combined_df_new.merge(stations_to_keep, on=["ST", "COUNTY", "STATION"], how="inner")

print(f"Original rows: {len(combined_df_new)}")
print(f"Final rows: {len(combined_df_final)}")
print(f"Unique counties: {combined_df_final.groupby(['ST', 'COUNTY']).ngroups}")
combined_df_final.head()

Original rows: 16314
Final rows: 11705
Unique counties: 1055


,STATION,NAME,LATITUDE,LONGITUDE,ELEVATION,DATE,CLDD,DT100,DT32,DX32,...,DX90,EMNT,EMXT,HTDD,PRCP,TAVG,TMAX,TMIN,COUNTY,ST
0,USW00025325,"KETCHIKAN AIRPORT, AK US",55.3586,-131.72196,25.7,2012,2.0,NaN,90.0,9.0,...,0.0,-18.9,27.2,4176.0,3655.8,6.9,9.7,4.1,KETCHIKAN GATEWAY BOROUGH,AK
1,USW00025325,"KETCHIKAN AIRPORT, AK US",55.3586,-131.72196,25.7,2013,8.0,NaN,63.0,7.0,...,0.0,-8.2,29.4,3901.3,3831.8,8.3,11.1,5.6,KETCHIKAN GATEWAY BOROUGH,AK
2,USW00025325,"KETCHIKAN AIRPORT, AK US",55.3586,-131.72196,25.7,2014,7.8,NaN,62.0,11.0,...,0.0,-12.1,26.1,3713.8,4258.5,8.4,11.4,5.3,KETCHIKAN GATEWAY BOROUGH,AK
3,USW00025325,"KETCHIKAN AIRPORT, AK US",55.3586,-131.72196,25.7,2015,15.4,NaN,44.0,4.0,...,0.0,-7.1,27.8,3246.7,4260.7,9.1,12.0,6.2,KETCHIKAN GATEWAY BOROUGH,AK
4,USW00025325,"KETCHIKAN AIRPORT, AK US",55.3586,-131.72196,25.7,2016,13.7,NaN,44.0,10.0,...,0.0,-9.3,25.6,3352.9,3551.8,9.1,12.1,6.2,KETCHIKAN GATEWAY BOROUGH,AK


In [21]:
census_tracts = pd.read_csv(os.path.join("cdc_data_raw", "CensusTractList2022.csv"))

census_tracts["CountyFIPS"] = (
    census_tracts["State code"].astype(str).str.zfill(2) +
    census_tracts["County code"].astype(str).str.zfill(3)
)

print(f"Census tract list shape: {census_tracts.shape}")
census_tracts.head()

Census tract list shape: (85528, 15)


,Year,MSA/MD code type,MSA/MD code,State code,County code,Tract,MSA/MD name,State,County name,FIPS code,MSA/MD MFI,Tract MFI,Tract income percentage,Tract income level,CountyFIPS
0,2022,MSA,10180,48,59,30101,"ABILENE, TX ...",TX,CALLAHAN COUNTY ...,48059030101,68388,61923.0,90.54,Middle,48059
1,2022,MSA,10180,48,59,30102,"ABILENE, TX ...",TX,CALLAHAN COUNTY ...,48059030102,68388,66132.0,96.70,Middle,48059
2,2022,MSA,10180,48,59,30200,"ABILENE, TX ...",TX,CALLAHAN COUNTY ...,48059030200,68388,59531.0,87.04,Middle,48059
3,2022,MSA,10180,48,253,20101,"ABILENE, TX ...",TX,JONES COUNTY ...,48253020101,68388,55179.0,80.68,Middle,48253
4,2022,MSA,10180,48,253,20102,"ABILENE, TX ...",TX,JONES COUNTY ...,48253020102,68388,0.0,0.00,Unknown,48253


In [26]:
# Build lookup: (State abbr, County name) -> CountyFIPS
# Strip " County" suffix from County name for better fuzzy matching
census_tracts["County name clean"] = (
    census_tracts["County name"]
    .str.replace(r'\s*county\s*', '', case=False, regex=True)
    .str.strip()
)

county_lookup = (
    census_tracts[["State", "County name clean", "CountyFIPS"]]
    .drop_duplicates()
    .dropna(subset=["State", "County name clean"])
)

# Get unique (ST, COUNTY) combos from combined_df_final
unique_counties = combined_df_final[["ST", "COUNTY"]].drop_duplicates().copy()

# Fuzzy match COUNTY to County name within the same state
def fuzzy_match_county(row, lookup_df, threshold=80):
    st = row["ST"]
    county = row["COUNTY"]
    
    # Filter lookup to same state
    state_counties = lookup_df[lookup_df["State"] == st]["County name clean"].tolist()
    
    if not state_counties:
        return None, None
    
    match, score, _ = process.extractOne(county, state_counties, scorer=fuzz.token_sort_ratio)
    
    if score >= threshold:
        fips = lookup_df[(lookup_df["State"] == st) & (lookup_df["County name clean"] == match)]["CountyFIPS"].values[0]
        return match, fips
    return None, None

print("Running fuzzy matching...")
unique_counties[["Matched County", "CountyFIPS"]] = unique_counties.apply(
    lambda row: pd.Series(fuzzy_match_county(row, county_lookup)),
    axis=1
)

print(f"Total unique counties: {len(unique_counties)}")
print(f"Matched: {unique_counties['CountyFIPS'].notna().sum()}")
print(f"Unmatched: {unique_counties['CountyFIPS'].isna().sum()}")

# Merge CountyFIPS back into combined_df_final
combined_df_final = combined_df_final.merge(
    unique_counties[["ST", "COUNTY", "Matched County", "CountyFIPS"]],
    on=["ST", "COUNTY"],
    how="left"
)

# Flag and export unmatched counties
unmatched = unique_counties[unique_counties["CountyFIPS"].isna()][["ST", "COUNTY"]]
unmatched.to_csv("unmatched_counties.csv", index=False)

print(f"\nUnmatched counties exported: {len(unmatched)}")
print(unmatched)
combined_df_final[["ST", "COUNTY", "Matched County", "CountyFIPS"]].drop_duplicates().head(10)

Running fuzzy matching...
Total unique counties: 1055
Matched: 1005
Unmatched: 50

Unmatched counties exported: 50
       ST                           COUNTY
12     AK            SKAGWAY-HOONAH-ANGOON
1000   VI                        ST. CROIX
1131   AK                           DENALI
2158   FL                             DADE
3410   LA                         ST. MARY
3413   LA                      PLAQUEMINES
3421   LA                        JEFFERSON
3433   LA                          ORLEANS
3445   LA                           IBERIA
3457   LA                        CALCASIEU
3469   LA                        LAFAYETTE
3493   LA                           VERNON
3496   LA                          RAPIDES
3508   LA                           SABINE
3513   LA                            CADDO
3525   LA                         OUACHITA
6163   SD                          SHANNON
7226   AK  PRINCE OF WALES-OUTER KETCHIKAN
7238   AK              WRANGELL-PETERSBURG
7250   AK                

,ST,COUNTY,Matched County,CountyFIPS
0,AK,KETCHIKAN GATEWAY BOROUGH,KETCHIKAN GATEWAY BOROUGH,02130
12,AK,SKAGWAY-HOONAH-ANGOON,None,None
24,AL,SHELBY,SHELBY,01117
36,AL,COLBERT,COLBERT,01033
48,AL,TUSCALOOSA,TUSCALOOSA,01125
60,AR,MONROE,MONROE,05095
65,AZ,MOHAVE,MOHAVE,04015
77,AZ,APACHE,APACHE,04001
88,CA,FRESNO,FRESNO,06019
100,CA,TEHAMA,TEHAMA,06103


In [27]:
# Import manually updated unmatched counties
unmatched_updated = pd.read_csv("unmatched_counties_updated.csv")

# Merge updated matches into combined_df_final
combined_df_final = combined_df_final.merge(
    unmatched_updated[["ST", "COUNTY", "Matched County", "CountyFIPS"]],
    on=["ST", "COUNTY"],
    how="left",
    suffixes=("", "_updated")
)

# Fill in missing Matched County and CountyFIPS with updated values
combined_df_final["Matched County"] = combined_df_final["Matched County"].fillna(combined_df_final["Matched County_updated"])
combined_df_final["CountyFIPS"] = combined_df_final["CountyFIPS"].fillna(combined_df_final["CountyFIPS_updated"])

# Drop temporary update columns
combined_df_final = combined_df_final.drop(columns=["Matched County_updated", "CountyFIPS_updated"])

# Remove rows with no matched CountyFIPS
before = len(combined_df_final)
combined_df_final = combined_df_final.dropna(subset=["CountyFIPS"])

print(f"Rows before: {before}")
print(f"Rows after dropping unmatched: {len(combined_df_final)}")
print(f"Rows dropped: {before - len(combined_df_final)}")
print(f"Unique counties: {combined_df_final.groupby(['ST', 'COUNTY']).ngroups}")
combined_df_final[["ST", "COUNTY", "Matched County", "CountyFIPS"]].drop_duplicates().head(10)

Rows before: 11705
Rows after dropping unmatched: 11594
Rows dropped: 111
Unique counties: 1044


,ST,COUNTY,Matched County,CountyFIPS
0,AK,KETCHIKAN GATEWAY BOROUGH,KETCHIKAN GATEWAY BOROUGH,02130
12,AK,SKAGWAY-HOONAH-ANGOON,SKAGWAY MUNICIPALITY ...,2230.0
24,AL,SHELBY,SHELBY,01117
36,AL,COLBERT,COLBERT,01033
48,AL,TUSCALOOSA,TUSCALOOSA,01125
60,AR,MONROE,MONROE,05095
65,AZ,MOHAVE,MOHAVE,04015
77,AZ,APACHE,APACHE,04001
88,CA,FRESNO,FRESNO,06019
100,CA,TEHAMA,TEHAMA,06103


In [28]:
combined_df_final.to_csv("combined_weather_final.csv", index=False)
print(f"Exported {len(combined_df_final)} rows to combined_weather_final.csv")

Exported 11594 rows to combined_weather_final.csv
